# 📈 Attrition Risk Dashboard (Notebook Version)
Since localhost ports are blocked, this notebook replicates the dashboard.

**What this does:**
1. Loads the master data and engineers features
2. Trains a fresh XGBoost model (avoids NumPy version mismatch)
3. Shows 2025 accuracy metrics (AUC, Recall, Precision)
4. Generates **2026 risk scores** for current employees
5. Lists **Saveable Stars** and **High Risk Employees**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from IPython.display import display, HTML

# Add project root
project_root = os.path.dirname(os.path.abspath('__file__'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import (
    PATH_OUTPUT, PATH_MODEL_OUTPUT, BASE_FEATURES,
    CATEGORICAL_FEATURES, TARGET_COLUMN, PREDICTION_HORIZON_MONTHS
)
from feature_engine import prepare_model_data

plt.style.use('ggplot')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)
print('✅ Libraries loaded.')

---
## 1. Load Master Data

In [ ]:
data_path = os.path.join(PATH_OUTPUT, 'Model_Ready_Data.parquet')
df = pd.read_parquet(data_path)
df['Snapshot_Date'] = pd.to_datetime(df['Snapshot_Date'])
print(f'✅ Data loaded: {len(df):,} rows  |  Years: {sorted(df["Year"].unique())}')
print(f'   Unique employees: {df["Employee_ID"].nunique():,}')
print(f'   Target column: {TARGET_COLUMN}')
print(f'   Positive targets by year:')
display(df.groupby('Year')[TARGET_COLUMN].sum().reset_index().rename(columns={TARGET_COLUMN: 'Leavers_Within_Horizon'}))

---
## 2. Train Fresh XGBoost Model (2025 Validation)
We train on data **before 2025** and test on **2025** to get accuracy metrics.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report

# Prepare 2025 validation split
X_train, y_train, X_test_2025, y_test_2025, test_df_2025, feat_names, _ = prepare_model_data(df, prediction_year=2025)

# Class imbalance weight
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos = n_neg / max(n_pos, 1)

# Train XGBoost
print('\n🚀 Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric='auc', random_state=42, use_label_encoder=False
)
xgb.fit(X_train, y_train, verbose=False)
print('✅ Model trained.')

---
## 3. Model Accuracy on 2025 (The "Exam")

In [ ]:
# Predict on 2025 test set
y_prob_2025 = xgb.predict_proba(X_test_2025)[:, 1]
threshold = 0.5
y_pred_2025 = (y_prob_2025 > threshold).astype(int)

auc = roc_auc_score(y_test_2025, y_prob_2025)
recall = recall_score(y_test_2025, y_pred_2025, zero_division=0)
precision = precision_score(y_test_2025, y_pred_2025, zero_division=0)
f1 = f1_score(y_test_2025, y_pred_2025, zero_division=0)

print('=' * 50)
print('   2025 VALIDATION RESULTS')
print('=' * 50)
print(f'   AUC:       {auc:.4f}')
print(f'   Recall:    {recall:.4f}  (% of actual leavers we caught)')
print(f'   Precision: {precision:.4f}  (% of flagged who actually left)')
print(f'   F1:        {f1:.4f}')
print(f'\n   Test Population: {len(X_test_2025):,}')
print(f'   Actual Leavers:  {int(y_test_2025.sum()):,}')
print(f'   Flagged by Model: {int(y_pred_2025.sum()):,}')
print('=' * 50)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_2025, y_pred_2025)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (2025)')

# Risk Distribution
axes[1].hist(y_prob_2025[y_test_2025 == 0], bins=50, alpha=0.6, label='Stayers', color='teal')
axes[1].hist(y_prob_2025[y_test_2025 == 1], bins=50, alpha=0.6, label='Leavers', color='salmon')
axes[1].axvline(threshold, color='red', linestyle='--', label=f'Threshold {threshold}')
axes[1].set_xlabel('Predicted Risk Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Risk Distribution: Stayers vs Leavers')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## 4. Feature Importance: Top Drivers of Attrition

In [ ]:
imps = pd.DataFrame({
    'Feature': feat_names,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=imps, palette='viridis')
plt.title('Top 15 Drivers of Attrition')
plt.xlabel('Feature Importance (Gain)')
plt.tight_layout()
plt.show()

---
## 5. 🚀 2026 Risk Profile (Current Employees)
Now we apply the trained model to the **latest 2026 snapshots** to get live risk scores.

In [ ]:
# Prepare 2026 data (train on everything before 2026)
_, _, X_test_2026, y_test_2026, test_df_2026, feat_names_2026, _ = prepare_model_data(df, prediction_year=2026)

# Align features
for c in X_train.columns:
    if c not in X_test_2026.columns:
        X_test_2026[c] = 0
X_test_2026 = X_test_2026[X_train.columns].fillna(0)

# Score
y_prob_2026 = xgb.predict_proba(X_test_2026)[:, 1]
test_df_2026 = test_df_2026.copy()
test_df_2026['Risk_Score'] = y_prob_2026
test_df_2026['Risk_Category'] = pd.cut(y_prob_2026, bins=[0, 0.3, 0.6, 1.0], labels=['🟢 Low', '🟡 Medium', '🔴 High'])
test_df_2026['Predicted_Leaver'] = (y_prob_2026 > threshold).astype(int)

print(f'✅ Risk scores generated for {len(test_df_2026):,} active employees (2026).')
print(f'\n   Risk Breakdown:')
display(test_df_2026['Risk_Category'].value_counts().reset_index().rename(columns={'index': 'Category', 'Risk_Category': 'Count'}))

In [ ]:
# Risk Distribution for 2026
plt.figure(figsize=(10, 4))
sns.histplot(test_df_2026['Risk_Score'], bins=50, kde=True, color='teal')
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold {threshold}')
plt.title('2026 Employee Risk Score Distribution')
plt.xlabel('Attrition Risk (6-month window)')
plt.ylabel('Employee Count')
plt.legend()
plt.show()

---
## 6. 📋 Action Lists

In [ ]:
# Display columns
show_cols = ['Employee_ID', 'Risk_Score', 'Risk_Category', 'Management_Level_Agg',
             'Total_Tenure_Years', 'Performance_Score', 'Level 2 Org', 'Level 3 Org']
show_cols = [c for c in show_cols if c in test_df_2026.columns]

# ---- SAVEABLE STARS ----
print('⭐ SAVEABLE STARS — High Performers at High Risk')
print('These are your top talent (Performance 4-5) who the model thinks may leave.\n')
stars = test_df_2026[
    (test_df_2026['Risk_Score'] > 0.5) &
    (test_df_2026.get('Performance_Score', pd.Series(dtype=float)).isin([4, 5]))
].sort_values('Risk_Score', ascending=False)

if not stars.empty:
    print(f'Total Saveable Stars: {len(stars)}')
    display(stars[show_cols].head(30))
else:
    print('No saveable stars found with current threshold.')

In [ ]:
# ---- HIGH RISK LIST ----
print('🔴 HIGH RISK EMPLOYEES (Risk > 0.7)')
print('These employees have a >70% chance of leaving within 6 months.\n')
high_risk = test_df_2026[test_df_2026['Risk_Score'] > 0.7].sort_values('Risk_Score', ascending=False)

if not high_risk.empty:
    print(f'Total High Risk: {len(high_risk)}')
    display(high_risk[show_cols].head(30))
else:
    print('No employees flagged with risk > 0.7.')

In [ ]:
# ---- RISK BY MANAGEMENT LEVEL ----
if 'Management_Level_Agg' in test_df_2026.columns:
    print('📊 RISK BY MANAGEMENT LEVEL')
    level_risk = test_df_2026.groupby('Management_Level_Agg').agg(
        Headcount=('Employee_ID', 'count'),
        Avg_Risk=('Risk_Score', 'mean'),
        High_Risk_Count=('Predicted_Leaver', 'sum'),
    ).reset_index()
    level_risk['High_Risk_Pct'] = (level_risk['High_Risk_Count'] / level_risk['Headcount'] * 100).round(1)
    level_risk = level_risk.sort_values('Avg_Risk', ascending=False)
    display(level_risk)

In [ ]:
# ---- RISK BY ORG ----
org_col = 'Level 2 Org' if 'Level 2 Org' in test_df_2026.columns else None
if org_col:
    print(f'📊 RISK BY {org_col.upper()}')
    org_risk = test_df_2026.groupby(org_col).agg(
        Headcount=('Employee_ID', 'count'),
        Avg_Risk=('Risk_Score', 'mean'),
        High_Risk_Count=('Predicted_Leaver', 'sum'),
    ).reset_index()
    org_risk['High_Risk_Pct'] = (org_risk['High_Risk_Count'] / org_risk['Headcount'] * 100).round(1)
    org_risk = org_risk.sort_values('Avg_Risk', ascending=False)
    display(org_risk)

In [ ]:
# ---- EXPORT TO CSV ----
export_path = os.path.join(PATH_MODEL_OUTPUT, 'Risk_Scores_2026.csv')
test_df_2026[show_cols].to_csv(export_path, index=False)
print(f'✅ Exported 2026 risk scores to: {export_path}')
print(f'   Total employees scored: {len(test_df_2026):,}')